## Importing Packages

In [1]:
import torch
import numpy as np
from tqdm import tqdm
import HAAMUNetPytorchCV as haam

2026-05-25 11:56:24,001 - INFO - Seed fixed for reproducibility.
2026-05-25 11:56:24,005 - WARNING - CUDA is not available. Using CPU.
2026-05-25 11:56:24,007 - INFO - Number of CPU cores available: 8
2026-05-25 11:56:24,009 - INFO - PyTorch version: 2.5.1
2026-05-25 11:56:24,011 - INFO - CUDA version: N/A



[INFO] Preparing DataLoaders for Fold 1/5...

[INFO] Preparing DataLoaders for Fold 2/5...

[INFO] Preparing DataLoaders for Fold 3/5...

[INFO] Preparing DataLoaders for Fold 4/5...

[INFO] Preparing DataLoaders for Fold 5/5...

[INFO] DataLoaders for all folds are ready!


## Create Test Dataset

In [ ]:
test_dataset = haam.BiomedicalDataset(
    img_dir="",
    mask_dir="",
    target_size=(256,256),
    augmentations=haam.get_validation_augmentations((256,256))
)
test_loader = haam.DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False
)

## Load Model Architecture and Weights

In [ ]:
device = torch.device("cpu")
print("Using device:", device)
model = haam.Att_UNet_HAAM_CNN(img_ch=3, output_ch=1)

model_path = ""

state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)

model = model.to(device)
model.eval()

print("Loaded HAAM CNN model")

Using device: cpu


C:\Users\hoda2\AppData\Local\Temp\ipykernel_6476\921846324.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location=device)


Loaded HAAM CNN model


## Inference

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

save_dir = ""
os.makedirs(save_dir, exist_ok=True)

with torch.no_grad():

    for batch in tqdm(test_loader):
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)
        mask_names = batch["mask_name"] 
        
        outputs = model(images)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float().cpu().numpy()

        for i in range(len(preds)):
            pred_mask = preds[i].squeeze() * 255
            pred_mask = pred_mask.astype(np.uint8)
            filename = os.path.basename(mask_names[i])
            save_path = os.path.join(save_dir, filename)
            cv2.imwrite(save_path, pred_mask)

## Calculate Metrices Validation

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
mask_dir = ""
pred_dir = ""

mask_files = sorted(os.listdir(mask_dir))
pred_files = sorted(os.listdir(pred_dir))

assert len(mask_files) == len(pred_files)

results = []

target_size = (256, 256)  

for mask_file, pred_file in zip(mask_files, pred_files):

    mask_path = os.path.join(mask_dir, mask_file)
    pred_path = os.path.join(pred_dir, pred_file)

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    pred = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)

    if mask is None or pred is None:
        print(f"Skipping {mask_file}")
        continue

    mask_resized = cv2.resize(mask, (target_size[0], target_size[1]), interpolation=cv2.INTER_NEAREST)

    mask_resized = (mask_resized > 127).astype(np.uint8)
    pred = (pred > 127).astype(np.uint8)

    mask_tensor = torch.from_numpy(mask_resized).float()
    pred_tensor = torch.from_numpy(pred).float()

    mask_tensor = mask_tensor.unsqueeze(0)
    pred_tensor = pred_tensor.unsqueeze(0)

    metrics = haam.calculate_metrics(mask_tensor, pred_tensor)

    results.append({
        "filename": mask_file,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1_score": metrics["f1_score"],
        "auc_roc": metrics["auc_roc"],
        "auc_pr": metrics["auc_pr"],
        "dice_similarity_coefficient": metrics["dice_similarity_coefficient"],
        "jaccard_index": metrics["jaccard_index"],
        "specificity": metrics["specificity"],
        "area_error_ratio": metrics["area_error_ratio"],
        "mean_absolute_error": metrics["mean_absolute_error"],
        "mean_intersection_over_union": metrics["mean_intersection_over_union"],
        "tversky_index": metrics["tversky_index"],
    })

df = pd.DataFrame(results)
df.to_csv("pred_metrics_val.csv", index=False)

print("Average Metrics:")
print(df.mean(numeric_only=True))